In [ ]:
from typing import Optional
import numpy as np
import gymnasium as gym
import math

In [ ]:
class FMCGEnv(gym.Env):

    def __init__(self,
                 p_min: int,
                 p_max: int,
                 p_diff: float,
                 alpha: float = 40.0,
                 beta: float = 2.0,
                 ):
        """ Initialize FMCG Gymnasium environment.

        Args:
            p_min (int): minimum price
            p_max (int): maximum price
            p_diff (float): price step
            alpha (float): demand core state param. (market size)
            beta (float): demand core state param. (price elasticity)
        """
        # main parameters
        self.p_min = p_min
        self.p_max = p_max
        self.p_diff = p_diff
        self.alpha = alpha
        self.beta = beta

        # check user input
        if not self._is_p_consistent(self.p_min, self.p_max, self.p_diff):
            raise ValueError(
                "Inconsistent price parameters, (p_max - p_min) / p_diff is not an integer."
            )

        # Define what the agent can observe
        # 3 times the maximum demand (lowest price) is set as max. demand
        max_d = 3 * int(self.alpha - self.beta * self.p_min)
        self.observation_space = gym.spaces.Discrete(max_d)
        
        # The available actions correspond to possible price points
        num_actions = round((self.p_max - self.p_min) / self.p_diff)
        self.action_space = gym.spaces.Discrete(num_actions)

        # Map action numbers to actual price points
        # rounding guards against floating point inconsistencies
        self._action_to_price = {
            i: round(self.p_min + i * self.p_diff, 10)
            for i in range(num_actions)
        }
    
    def _is_p_consistent(self,p_min:int,
                         p_max:int,
                         p_diff:float):
        """ Checks consistency of price parameters.
        
        Args:
            p_min (int): minimum price
            p_max (int): maximum price
            p_diff (float): price step

        Returns:
            bool indicating whether price params. are consistent
        """
        # to have an int. number of steps, the price range
        # should be divisible by p_diff
        n = (p_max - p_min) / p_diff
        return math.isclose(n, round(n), rel_tol=1e-9)